# Training de YOLOv8n para detección cenital de cabezas

Phase A del re-entreno del People Counter: fine-tune de `yolov8n` sobre el dataset Roboflow `overhead_person` (o equivalente), y export a ONNX para que después se pueda compilar a un `.hef` de Hailo-8L en un workstation (WSL2 + Hailo Dataflow Compiler). El notebook está pensado para correr en Kaggle Notebooks con GPU T4.

**Antes de correr:**

1. Sidebar derecho → **Settings** → **Accelerator** → **GPU T4 x2** (o GPU P100, lo que esté disponible).
2. Sidebar derecho → **Add-ons** → **Secrets** → **Add Secret** con:
   - Label: `ROBOFLOW_API_KEY`
   - Value: tu API key de Roboflow
   - Toggle **Attached** a este notebook → ON.
3. Editá `WORKSPACE`, `PROJECT`, `VERSION` en la cell 4 si vas a entrenar con un dataset distinto al default.
4. Sidebar derecho → **Run All**.

Wall-time esperado en T4: ~1-2 h para ~5k imágenes × 50 epochs con BATCH=8.

## 1. Setup de paths (Kaggle)

Kaggle expone tres directorios:
- `/kaggle/input/` — datasets attachados (read-only)
- `/kaggle/working/` — output del notebook (escribible, persistido)
- `/tmp/` — scratch (se borra al cerrar la sesión)

Vamos a guardar el dataset descargado en `/kaggle/working/dataset/` y los runs en `/kaggle/working/runs/`.

In [ ]:
import os
OUTPUT_ROOT = '/kaggle/working'
DATASET_ROOT = f'{OUTPUT_ROOT}/dataset'
RUNS_ROOT    = f'{OUTPUT_ROOT}/runs'
EXPORT_ROOT  = f'{OUTPUT_ROOT}/export'
for p in (DATASET_ROOT, RUNS_ROOT, EXPORT_ROOT):
    os.makedirs(p, exist_ok=True)
print('Workspace:', OUTPUT_ROOT)

## 2. Instalar dependencias

Kaggle ya trae `ultralytics` preinstalado en muchos kernels, pero lo pineamos a la versión 8.3.* por reproducibilidad.

In [ ]:
!pip install -q ultralytics==8.3.* roboflow==1.1.*
import ultralytics; ultralytics.checks()

## 3. Leer la API key desde Kaggle Secrets

En el sidebar derecho → **Add-ons** → **Secrets** debe estar `ROBOFLOW_API_KEY` con "Attached" en ON para este notebook.

In [ ]:
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
ROBOFLOW_API_KEY = secrets.get_secret('ROBOFLOW_API_KEY')
assert ROBOFLOW_API_KEY, 'Add ROBOFLOW_API_KEY a Kaggle Secrets y attach al notebook.'
print('API key cargada (length =', len(ROBOFLOW_API_KEY), ')')

## 4. Bajar el dataset de Roboflow

Editá los tres slugs para matchear la URL del dataset en Roboflow Universe:
`https://universe.roboflow.com/<WORKSPACE>/<PROJECT>/<VERSION>`

In [ ]:
WORKSPACE = 'coding-compass-nmjfb'
PROJECT   = 'overhead-head-detection-cwetj'
VERSION   = 2

from roboflow import Roboflow
rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace(WORKSPACE).project(PROJECT)
dataset = project.version(VERSION).download(
    'yolov8',
    location=f'{DATASET_ROOT}/{WORKSPACE}__{PROJECT}__v{VERSION}',
    overwrite=True,
)
DATA_YAML = f'{dataset.location}/data.yaml'
print('Dataset en:', dataset.location)
print('data.yaml :', DATA_YAML)
!cat "$DATA_YAML"

## 5. Entrenar

Arrancamos desde `yolov8n.pt` (pretrained en COCO). `BATCH=8` por defecto — el dataset tiene escenas con muchas cabezas por imagen y el TaskAlignedAssigner del loss es memory-intensive, así que conviene quedarse abajo de 16 para evitar OOM en el T4.

In [ ]:
EPOCHS = 50
BATCH  = 8         # subí a 16 si Kaggle T4 te da los 16GB completos sin OOM
IMGSZ  = 640
RUN_NAME = f'{PROJECT}_v{VERSION}_yolov8n'

from ultralytics import YOLO
model = YOLO('yolov8n.pt')

results = model.train(
    data=DATA_YAML,
    epochs=EPOCHS,
    batch=BATCH,
    imgsz=IMGSZ,
    project=RUNS_ROOT,
    name=RUN_NAME,
    exist_ok=True,
    save_period=10,
    patience=20,
    pretrained=True,
    optimizer='AdamW',
    lr0=0.001,
    seed=42,
    verbose=True,
)
BEST_PT = f'{RUNS_ROOT}/{RUN_NAME}/weights/best.pt'
print('Best weights en:', BEST_PT)

## 6. Validar

In [ ]:
from ultralytics import YOLO
best = YOLO(BEST_PT)
metrics = best.val(data=DATA_YAML, imgsz=IMGSZ, plots=True)
print('mAP50    :', float(metrics.box.map50))
print('mAP50-95 :', float(metrics.box.map))
print('Precision:', float(metrics.box.mp))
print('Recall   :', float(metrics.box.mr))

## 7. Exportar a ONNX

In [ ]:
best.export(format='onnx', imgsz=IMGSZ, opset=12, dynamic=False, simplify=True)
ONNX_PATH = BEST_PT.replace('.pt', '.onnx')
print('ONNX:', ONNX_PATH)
print('size (MB):', round(os.path.getsize(ONNX_PATH) / (1024*1024), 2))

## 8. Stash de outputs en `/kaggle/working/export/`

Copiamos `best.pt`, `best.onnx`, `data.yaml` y un calibration set de 200 imágenes a `/kaggle/working/export/<run-name>/`. Kaggle persiste todo lo que está en `/kaggle/working/` y al final de la sesión podés bajarlo con el botón **Output → Download All** del sidebar derecho.

In [ ]:
import shutil, random, glob
EXPORT_DIR = f'{EXPORT_ROOT}/{RUN_NAME}'
os.makedirs(EXPORT_DIR, exist_ok=True)

shutil.copy(ONNX_PATH, f'{EXPORT_DIR}/best.onnx')
shutil.copy(BEST_PT,   f'{EXPORT_DIR}/best.pt')
shutil.copy(DATA_YAML, f'{EXPORT_DIR}/data.yaml')

calib_dir = f'{EXPORT_DIR}/calib'
os.makedirs(calib_dir, exist_ok=True)
train_imgs = glob.glob(f'{dataset.location}/train/images/*')
random.seed(42)
for src in random.sample(train_imgs, min(200, len(train_imgs))):
    shutil.copy(src, calib_dir)

print('Stash en:', EXPORT_DIR)
!ls -la "$EXPORT_DIR"

## 9. Bajar los outputs al workstation

Kaggle preserva `/kaggle/working/` como output del notebook. Para descargar:

1. Sidebar derecho → **Output**.
2. Botón **Download All** → te baja un .zip con todo `/kaggle/working/`.
3. En tu workstation, descomprimís y vas a tener `/export/<RUN_NAME>/best.onnx` + `calib/` listos para `hailomz compile` en WSL2.

Próximo paso (off-Kaggle): compilar a HEF.

```bash
# Adentro de WSL2 con hailo-dataflow-compiler instalado
hailomz compile yolov8n \
    --ckpt best.onnx \
    --hw-arch hailo8l \
    --calib-path calib/
```

Output: `yolov8n.hef`. SCP a la Pi y actualizá `detection.model_path` en `/etc/people-counter/config.yaml`.